<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/dynatree-cre/notebooks/m7_4_colab_rerank_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M7.4 — Candidate-Aware Prompt Reranking (`DT-C2.1` Flat vs `DT-C4.4` Tree) Benchmark (Colab GPU, Seed 2021)
- So sánh công bằng **Flat Candidate Reranking (`DT-C2.1`)** vs **Tree Candidate Reranking (`DT-C4.4`)**.
- Với mỗi run, `--rerank-sweep` xuất toàn bộ bảng **12 cấu hình**: $K \in \{1, 3, 5, 10\} \times \{\text{retrieval\_only}, \text{prompted\_only}, \text{fusion}\}$ kèm `Candidate Recall@K`, `Conditional Rerank Acc`, `Final Acc`, `Forgetting`, `Latency`, `Peak VRAM` và `BERT forward passes`.
- Chạy trên FewRel Standard, Exp B (`semantic_homogeneous`), Exp C (`cross_task_dispersed`) và TACRED.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chon GPU runtime (T4/L4/A100)!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/dynatree-cre'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scipy scikit-learn tabulate pytest
!python scripts/setup_protocol_v1_data.py

gdrive_runs = Path('/content/drive/MyDrive/TAPTA_DynaTree_CRE/runs/dynatree_cre')
gdrive_runs.mkdir(parents=True, exist_ok=True)
local_runs = Path('/content/research-code-lab/runs/dynatree_cre')
local_runs.parent.mkdir(parents=True, exist_ok=True)
if local_runs.is_symlink():
    local_runs.unlink()
elif local_runs.exists():
    shutil.copytree(local_runs, gdrive_runs, dirs_exist_ok=True)
    shutil.rmtree(local_runs)
local_runs.symlink_to(gdrive_runs)
!pytest tests/dynatree_cre -q
print('Setup & Drive backup ready ->', gdrive_runs)

In [ ]:
# Part A — FewRel Standard (Seed 2021): DT-C2.1 Flat Rerank vs DT-C4.4 Tree Rerank
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_4_dt_c2_1_flat_rerank_fewrel_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_4_dt_c4_4_tree_rerank_fewrel_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

In [ ]:
# Part B — FewRel Stress Tests (Exp B semantic_homogeneous & Exp C cross_task_dispersed)
for dist, tag in [('semantic_homogeneous', 'expB'), ('cross_task_dispersed', 'expC')]:
    !python experiments/run_dynatree_cre.py \
        --config configs/dynatree_cre/m7_4_dt_c2_1_flat_rerank_fewrel_5shot_seed2021.yaml \
        --task-distribution {dist} \
        --output-dir runs/dynatree_cre/m7_4_dt_c2_1_{tag}/fewrel_5shot/seed_2021 \
        --rerank-sweep --device cuda --allow-hub-download
    !python experiments/run_dynatree_cre.py \
        --config configs/dynatree_cre/m7_4_dt_c4_4_tree_rerank_fewrel_5shot_seed2021.yaml \
        --task-distribution {dist} \
        --output-dir runs/dynatree_cre/m7_4_dt_c4_4_{tag}/fewrel_5shot/seed_2021 \
        --rerank-sweep --device cuda --allow-hub-download

In [ ]:
# Part C — TACRED (8 tasks, Seed 2021): DT-C2.1 Flat vs DT-C4.4 Tree Rerank
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_4_dt_c2_1_flat_rerank_tacred_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_4_dt_c4_4_tree_rerank_tacred_5shot_seed2021.yaml \
    --rerank-sweep --device cuda --allow-hub-download

In [ ]:
# Part D — Summary Table (K=1,3,5,10 x Fusion Modes) & Export Bundle
import json
import os
import shutil
from pathlib import Path
from tabulate import tabulate
from google.colab import files

runs_root = Path('/content/research-code-lab/runs/dynatree_cre')
bundle = Path('/content/m7_4_bundle')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)

rows = []
for m_path in sorted(runs_root.glob('m7_4_*/**/metrics.json')):
    run_dir = m_path.parent
    rel = run_dir.relative_to(runs_root)
    shutil.copytree(run_dir, bundle / rel, ignore=shutil.ignore_patterns('*.pt'), dirs_exist_ok=True)
    d = json.loads(m_path.read_text())
    res = json.loads((run_dir / 'resource_metrics.json').read_text())
    sw_path = run_dir / 'rerank_sweep.json'
    best_cell = {}
    if sw_path.exists():
        sw = json.loads(sw_path.read_text())
        best_cell = max(sw['cells'], key=lambda c: c['final_ncm_whole_accuracy'])
    rr = d['stages'][-1]['eval_stats'].get('rerank_diagnostics', {})
    rows.append([
        rel.parts[0],
        d.get('dataset'),
        d.get('task_distribution'),
        f"{d.get('final_ncm_whole_accuracy', 0)*100:.2f}%",
        f"{d.get('ncm_summary', {}).get('average_forgetting', 0)*100:.2f}%",
        f"{rr.get('candidate_recall_at_k', 0)*100:.2f}%",
        f"{rr.get('conditional_rerank_ncm_acc', 0)*100:.2f}%",
        f"{best_cell.get('final_ncm_whole_accuracy', 0)*100:.2f}% (K={best_cell.get('k')}, {best_cell.get('fusion_mode')})" if best_cell else 'N/A',
        f"{res.get('final_inference_latency_ms_per_sample', 0):.1f}ms",
        f"{res.get('peak_memory_bytes', 0)/(1024**2):.0f}MB",
    ])

headers = ['Run', 'Dataset', 'Dist', 'K=5 Fusion NCM', 'NCM F', 'R@5', 'Cond@5', 'Best Sweep Cell', 'Lat', 'PeakVRAM']
print(tabulate(rows, headers=headers, tablefmt='grid'))

shutil.make_archive('/content/m7_4_colab_results', 'zip', bundle)
print('Bundle size KB:', os.path.getsize('/content/m7_4_colab_results.zip') / 1024)
files.download('/content/m7_4_colab_results.zip')